# General Relativity

This notebook covers 7 equations from Einstein's theory of General Relativity: geodesic motion in curved spacetime, cosmological evolution, gravitational wave propagation, and neutron star structure.

Each section presents:
1. The defining equation with physical context
2. A symbolic solution
3. A numerical solution with visualization

All equations are registered in the `diff_eq_solver` equation registry.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from diff_eq_solver import registry
from diff_eq_solver.visualizer import plot_ode_solution, plot_phase_portrait, plot_pde_heatmap, plot_pde_snapshots, plot_3d_surface, plot_comparison, plot_orbit, plot_special_function
from IPython.display import Math, display

## 1. Schwarzschild Geodesic

The geodesic equation in Schwarzschild spacetime describes orbital motion around a spherically symmetric mass $M$. For the equatorial plane ($\theta = \pi/2$), the radial equation is:

$$\frac{d^2u}{d\phi^2} + u = \frac{M}{L^2} + 3Mu^2$$

where $u = 1/r$, $L$ is the specific angular momentum, and the $3Mu^2$ term is the relativistic correction that produces perihelion precession.

**Applications:** Mercury's perihelion advance, binary pulsar orbits, black hole accretion disk dynamics.

In [ ]:
# Symbolic solution
eq = registry.get('schwarzschild_geodesic')
print("Schwarzschild Geodesic Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: precessing ellipse orbit
# Use geometric units M=1; choose L and E for bound orbit
# Initial conditions: u(0) = 1/r_perihelion, u'(0) = 0

M = 1.0
L = 4.0  # specific angular momentum (must be > sqrt(3)*M*sqrt(3) for stable orbit)

# Perihelion at r_min ~ 8M for near-circular orbit with L=4
r_peri = 8.0 * M
u0 = 1.0 / r_peri
du0 = 0.0  # turning point

phi_span = (0, 20 * np.pi)  # multiple orbits to show precession
phi_eval = np.linspace(0, 20 * np.pi, 5000)

result = eq.numerical_solve(phi_span, [u0, du0], t_eval=phi_eval,
    params={'M': M, 'L': L})

# Convert u -> r and phi -> (x,y)
u_vals = result.y[0]
phi_vals = result.t
r_vals = 1.0 / u_vals
x_vals = r_vals * np.cos(phi_vals)
y_vals = r_vals * np.sin(phi_vals)

fig, ax = plt.subplots(figsize=(8, 8))
plot_orbit(x_vals, y_vals, ax=ax)

# Mark central mass
ax.plot(0, 0, 'ko', markersize=8, label='Central mass $M$')
ax.set_title('Schwarzschild Geodesic: Precessing Ellipse')
ax.set_xlabel('$x$ (geometric units)')
ax.set_ylabel('$y$ (geometric units)')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Friedmann Equations

The Friedmann equations govern the expansion of a homogeneous, isotropic universe:

$$\left(\frac{\dot{a}}{a}\right)^2 = \frac{8\pi G}{3}\rho - \frac{k}{a^2} + \frac{\Lambda}{3}$$

$$\frac{\ddot{a}}{a} = -\frac{4\pi G}{3}(\rho + 3p) + \frac{\Lambda}{3}$$

where $a(t)$ is the scale factor, $\rho$ is energy density, $k$ is spatial curvature, and $\Lambda$ is the cosmological constant.

**Applications:** Big Bang cosmology, dark energy, cosmic microwave background, large-scale structure formation.

In [ ]:
# Symbolic solution
eq = registry.get('friedmann_equations')
print("Friedmann Equations:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: plot a(t) for three cosmologies
fig, ax = plt.subplots(figsize=(10, 6))
t_span = (0.01, 1.5)
t_eval = np.linspace(0.01, 1.5, 500)

cosmologies = [
    {'Omega_m': 1.0, 'Omega_Lambda': 0.0, 'label': 'Matter-only ($\Omega_m=1$)',
     'color': '#1f77b4', 'linestyle': '-'},
    {'Omega_m': 0.3, 'Omega_Lambda': 0.7, 'label': '$\Lambda$CDM ($\Omega_m=0.3, \Omega_\Lambda=0.7$)',
     'color': '#ff7f0e', 'linestyle': '-'},
    {'Omega_m': 0.0, 'Omega_Lambda': 1.0, 'label': 'de Sitter ($\Omega_\Lambda=1$)',
     'color': '#2ca02c', 'linestyle': '-'},
]

for cosmo in cosmologies:
    label = cosmo.pop('label')
    color = cosmo.pop('color')
    ls = cosmo.pop('linestyle')
    # a(0) near 0, a'(0) set by H0
    y0 = [0.01, 0.01]  # a(t0), a'(t0)
    result = eq.numerical_solve(t_span, y0, t_eval=t_eval, params=cosmo)
    plot_special_function(result, ax=ax, label=label, color=color, linestyle=ls)

ax.set_title('Friedmann Equation: Scale Factor $a(t)$ for Different Cosmologies')
ax.set_xlabel('Cosmic time $t$')
ax.set_ylabel('Scale factor $a(t)$')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Gravitational Wave (Linearized)

In the weak-field limit, gravitational waves satisfy the linearized wave equation:

$$\frac{\partial^2 h}{\partial t^2} - c^2 \frac{\partial^2 h}{\partial x^2} = 0$$

where $h_{\mu\nu}$ is the metric perturbation. For the $+$ polarization:
$h_+(t, z) = A \cos(\omega t - kz)$.

**Applications:** LIGO/Virgo/KAGRA gravitational wave detection, binary black hole mergers, neutron star mergers, cosmological backgrounds.

In [ ]:
# Symbolic solution
eq = registry.get('gravitational_wave_linearized')
print("Linearized Gravitational Wave Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: wave propagation heatmap
# Set up a Gaussian pulse initial condition
Nx = 200
Nt = 300
x = np.linspace(-10, 10, Nx)
t = np.linspace(0, 6, Nt)

# Initial condition: Gaussian pulse
h0 = np.exp(-x**2 / 2.0)
dh0 = np.zeros(Nx)  # initially at rest

result = eq.numerical_solve(
    t_span=(0, 6),
    u0=np.column_stack([h0, dh0]),
    x=x,
    t_eval=t,
    params={'c': 1.0}
)

fig, ax = plt.subplots(figsize=(10, 6))
plot_pde_heatmap(result, ax=ax, x_label='$z$', t_label='$t$', title='Gravitational Wave Propagation: $h_+(t,z)$')
plt.tight_layout()
plt.show()

## 4. Tolman-Oppenheimer-Volkoff (TOV) Equation

The TOV equation describes the structure of a spherically symmetric body in hydrostatic equilibrium in general relativity:

$$\frac{dP}{dr} = -\frac{G}{r^2}\left(\rho + \frac{P}{c^2}\right)\left(m + \frac{4\pi r^3 P}{c^2}\right)\left(1 - \frac{2Gm}{rc^2}\right)^{-1}$$

along with $dm/dr = 4\pi r^2 \rho$.

**Applications:** Neutron star structure, maximum mass limits, equation of state constraints, compact object astrophysics.

In [ ]:
# Symbolic solution
eq = registry.get('tov_equation')
print("Tolman-Oppenheimer-Volkoff Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: P(r) and m(r) for a neutron star
# Use polytropic EOS: P = K * rho^gamma

# Central pressure and mass at r=0
P_center = 1.0e-3  # in geometric units
m_center = 0.0
r_span = (0.01, 15.0)
r_eval = np.linspace(0.01, 15.0, 1000)

result = eq.numerical_solve(r_span, [P_center, m_center], t_eval=r_eval,
    params={'K': 100.0, 'gamma': 2.0})

r_vals = result.t
P_vals = result.y[0]
m_vals = result.y[1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pressure profile
ax1.plot(r_vals, P_vals, 'b-', linewidth=2)
ax1.set_xlabel('Radius $r$')
ax1.set_ylabel('Pressure $P(r)$')
ax1.set_title('TOV: Pressure Profile')
ax1.grid(True, alpha=0.3)
ax1.axhline(0, color='black', linewidth=0.5)

# Mass profile
ax2.plot(r_vals, m_vals, 'r-', linewidth=2)
ax2.set_xlabel('Radius $r$')
ax2.set_ylabel('Enclosed mass $m(r)$')
ax2.set_title('TOV: Mass Profile')
ax2.grid(True, alpha=0.3)

plt.suptitle('Neutron Star Structure (TOV Equation)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Kerr Geodesic

The geodesic equation in Kerr spacetime describes motion around a rotating black hole with mass $M$ and spin parameter $a = J/M$. For equatorial orbits:

$$\Sigma \frac{dr}{d\lambda} = \pm\sqrt{R(r)}$$

where $\Sigma = r^2 + a^2\cos^2\theta$ and $R(r)$ is the radial potential incorporating frame-dragging effects.

**Applications:** Frame-dragging (Lense-Thirring effect), black hole spin measurement, accretion disk physics, ergosphere dynamics.

In [ ]:
# Symbolic solution
eq = registry.get('kerr_geodesic')
print("Kerr Geodesic Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: equatorial orbit in Kerr spacetime
M = 1.0
a = 0.5 * M  # spin parameter (0 < a < M)
L = 3.8       # specific angular momentum

# Solve in (r, phi) coordinates
r_init = 10.0 * M
phi_span = (0, 25 * np.pi)
phi_eval = np.linspace(0, 25 * np.pi, 6000)

# u = 1/r, du/dphi = 0 at perihelion
u0 = 1.0 / r_init
du0 = 0.0

result = eq.numerical_solve(phi_span, [u0, du0], t_eval=phi_eval,
    params={'M': M, 'a': a, 'L': L})

u_vals = result.y[0]
phi_vals = result.t
r_vals = 1.0 / u_vals
x_vals = r_vals * np.cos(phi_vals)
y_vals = r_vals * np.sin(phi_vals)

fig, ax = plt.subplots(figsize=(8, 8))
plot_orbit(x_vals, y_vals, ax=ax)

# Draw ergosphere and event horizon
r_plus = M + np.sqrt(M**2 - a**2)  # outer horizon
r_ergo = 2 * M  # ergosphere in equatorial plane
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(r_plus * np.cos(theta), r_plus * np.sin(theta), 'k--',
        linewidth=1.5, label=f'Event horizon ($r_+ = {r_plus:.2f}M$)')
ax.plot(r_ergo * np.cos(theta), r_ergo * np.sin(theta), 'r--',
        linewidth=1.5, label=f'Ergosphere ($r_{{ergo}} = {r_ergo:.1f}M$)')

ax.set_title(f'Kerr Geodesic: Equatorial Orbit ($a/M = {a/M:.1f}$)')
ax.set_xlabel('$x$ (geometric units)')
ax.set_ylabel('$y$ (geometric units)')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Robertson-Walker Cosmology

The Robertson-Walker metric describes the geometry of a homogeneous, isotropic universe. The evolution of density components follows from the continuity equation:

$$\dot{\rho}_i + 3H(\rho_i + p_i) = 0$$

For each component: $\rho_m \propto a^{-3}$, $\rho_r \propto a^{-4}$, $\rho_\Lambda = \text{const}$.

**Applications:** Component energy density evolution, matter-radiation equality, cosmological horizon, nucleosynthesis.

In [ ]:
# Symbolic solution
eq = registry.get('robertson_walker_cosmology')
print("Robertson-Walker Cosmology:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: plot Omega_m, Omega_r, Omega_Lambda vs a
# Use the Friedmann equation to compute H(a), then density fractions

a_eval = np.linspace(0.01, 3.0, 500)

# LambdaCDM parameters
Omega_m0 = 0.3
Omega_r0 = 9.0e-5  # radiation (photons + neutrinos)
Omega_Lambda0 = 0.7
H0 = 67.4  # km/s/Mpc (Planck 2018)

result = eq.numerical_solve(
    a_span=(0.01, 3.0),
    y0=[Omega_m0, Omega_r0, Omega_Lambda0],
    t_eval=a_eval,
    params={'H0': H0}
)

a_vals = result.t
# Compute density parameters
Omega_m = Omega_m0 * a_vals**(-3)
Omega_r = Omega_r0 * a_vals**(-4)
Omega_L = np.full_like(a_vals, Omega_Lambda0)
Omega_total = Omega_m + Omega_r + Omega_L

# Normalize to get Omega_i
Omega_m_frac = Omega_m / Omega_total
Omega_r_frac = Omega_r / Omega_total
Omega_L_frac = Omega_L / Omega_total

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(a_vals, Omega_m_frac, 'b-', linewidth=2, label='$\Omega_m$ (matter)')
ax.plot(a_vals, Omega_r_frac, 'r-', linewidth=2, label='$\Omega_r$ (radiation)')
ax.plot(a_vals, Omega_L_frac, 'g-', linewidth=2, label='$\Omega_\Lambda$ (dark energy)')

# Mark matter-radiation equality and matter-Lambda equality
a_eq_mr = (Omega_r0 / Omega_m0)  # matter-radiation equality
a_eq_ml = (Omega_m0 / Omega_Lambda0)**(1.0/3.0)  # matter-lambda equality
ax.axvline(a_eq_mr, color='purple', linestyle=':', alpha=0.5, label=f'$a_{{mr}} = {a_eq_mr:.4f}$')
ax.axvline(a_eq_ml, color='orange', linestyle=':', alpha=0.5, label=f'$a_{{m\Lambda}} = {a_eq_ml:.2f}$')

ax.set_xlabel('Scale factor $a$')
ax.set_ylabel('Density parameter $\Omega_i$')
ax.set_title('Robertson-Walker Cosmology: Component Evolution')
ax.legend(loc='center right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0.01, 3.0)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 7. de Sitter Cosmology

The de Sitter solution describes a universe dominated entirely by the cosmological constant $\Lambda$:

$$\frac{\ddot{a}}{a} = \frac{\Lambda}{3}, \qquad a(t) = a_0 \exp\left(\sqrt{\frac{\Lambda}{3}}\, t\right)$$

This gives exponential expansion with Hubble parameter $H = \sqrt{\Lambda/3}$.

**Applications:** Cosmic inflation (early universe), late-time dark energy domination, holographic principle, AdS/CFT correspondence.

In [ ]:
# Symbolic solution
eq = registry.get('desitter_cosmology')
print("de Sitter Cosmology:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: exponential expansion
Lambda_vals = [0.5, 1.0, 2.0]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig, ax = plt.subplots(figsize=(10, 6))
t_span = (0, 3.0)
t_eval = np.linspace(0, 3.0, 500)

for Lam, color in zip(Lambda_vals, colors):
    y0 = [1.0, np.sqrt(Lam / 3.0)]  # a(0)=1, a'(0)=H0
    result = eq.numerical_solve(t_span, y0, t_eval=t_eval,
        params={'Lambda': Lam})
    plot_special_function(result, ax=ax,
        label=f'$\Lambda = {Lam}$', color=color)

ax.set_title('de Sitter Cosmology: Exponential Expansion $a(t) \propto e^{Ht}$')
ax.set_xlabel('Time $t$')
ax.set_ylabel('Scale factor $a(t)$')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()